# Stoic Qwen3.5 Text 9B Fine-Tuning with Unsloth (BNB NF4 QLoRA)

**Base Model:** Qwen3.5 text-only 9B, pre-quantized BNB NF4 4-bit

**Dataset:** Per-persona datagen JSONL — each persona has its own system prompt and distinctive voice

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Deployment Target:** vLLM serving on an A5000 (24GB VRAM). The PEFT LoRA adapter is the artifact that ships; serve it with a compatible Qwen3.5 9B base after a smoke test. The GGUF export cell at the end is unused for this deployment but kept for ad-hoc reuse.

**Chat Template:** Tokenizer-native Qwen ChatML template (applied via `tokenizer.apply_chat_template`)

**Architecture:** This base LoRA teaches the model persona-switching — "when the system prompt says you're Marcus Aurelius, speak like Marcus; when it says Seneca, speak like Seneca." Optional persona LoRAs can refine individual voices further.

## 1. Configuration

All paths and variables for easy configuration.

In [ ]:
# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
PROJECT_ROOT = "/workspace/training/stoic"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== MODEL CONFIGURATION ===========================
# Text-only Qwen3.5 9B with the vision tower removed and weights pre-quantized
# to BNB NF4 4-bit. This is the clean QLoRA training base; AWQ is better kept
# for inference deployments, not SFT training.
BASE_LLM = "techwithsergiu/Qwen3.5-text-9B-bnb-4bit"
MODEL_NAME_BASE = "stoic_qwen35_9b_sft_techwithsergiu_bnb_4bit"

# =========================== INPUT DATA ===========================
# Combined v2 multi-turn ShareGPT JSONL (SFT) from datagen notebooks
# Already quality-filtered, multi-turn (4 QA pairs per conversation), grouped by topic
INPUT_DATA_FILE = f"{PROJECT_ROOT}/data/training-data/stoic_persona/stoic_personas_combined_sharegpt.jsonl"

# =========================== PERSONA SYSTEM PROMPTS ===========================
# System prompts are EXTRACTED from the JSONL at load time (see data loading cell below).
# This keeps training in sync with datagen — if you regenerate data with new/changed
# prompts, the training notebook picks them up automatically.
# After loading, the dict `persona_system_prompts` maps persona_key -> full prompt text.
# It is also saved alongside the LoRA adapters for use at inference time.

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH = 4096
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
TARGET_EPOCHS = 1
SFT_MAX_EXAMPLES = 3000  # 0 = use all valid conversations; set e.g. 500/1000/3000 for capped runs

# =========================== LoRA CONFIGURATION ===========================
# Adapter is merged into the base weights at GGUF export, so adapter size on
# disk is irrelevant. Use full attention + MLP targets and a higher rank for
# stronger persona-distinction learning.
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0

# Full "all-linear" Unsloth recipe: attention + MLP projections.
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "I am struggling with anxiety about things outside my control. How do I find peace?"

# ============================================================================
print("✓ Configuration loaded (Qwen3.5 text 9B BNB NF4 QLoRA, v2 SFT)")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_FILE}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  LoRA output: {LORA_OUTPUT_DIR}")
print(f"  LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training: batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"  SFT max examples: {SFT_MAX_EXAMPLES or 'ALL'}")
print(f"  Training precision: pre-quantized BNB NF4 4-bit QLoRA")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Persona prompts: extracted from JSONL at load time")

## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries.

In [ ]:
# Install core packages in the running notebook container
!pip install -q -U --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel ≥0.16 on PyPI, so
# uninstall — peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
!pip uninstall -y -q torchao

# Qwen3.5 support may be ahead of PyPI releases.
!pip install -q -U git+https://github.com/huggingface/transformers.git

# Keep PEFT compatible with latest Transformers main.
!pip install -q -U git+https://github.com/huggingface/peft.git

# Verify installations
import importlib.util
import unsloth
import unsloth_zoo
import transformers
import peft
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Unsloth-Zoo: {unsloth_zoo.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ PEFT: {peft.__version__}")
print(f"✓ TRL: {trl.__version__}")
print(f"✓ torchao installed: {importlib.util.find_spec('torchao') is not None} (should be False)")
print("Environment ready. Restart kernel, then rerun from Cell 3.")


## 3. Load Dataset

Load the combined multi-turn ShareGPT JSONL from `stoic_datagen.ipynb`.

- Already quality-filtered (no short answers, no AI refusals)
- Multi-turn: 4 QA pairs grouped per conversation by topic
- Each conversation has a persona-specific system prompt
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`

In [ ]:

import json, os, re
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING COMBINED SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

# Load multi-turn conversations and EXTRACT system prompts from the JSONL.
# This replaces hardcoded prompt dicts — prompts stay in sync with datagen automatically.
conversations = []
persona_system_prompts = {}   # persona_key -> full system prompt text
persona_counts = defaultdict(int)

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)
        conversations.append(conv)

        # Extract persona name from "You are <Name>, ..." pattern
        sys_msg = conv["conversations"][0]["value"]
        match = re.match(r"You are (.+?),", sys_msg)
        if match:
            raw_name = match.group(1)
            # Normalize to snake_case key: lowercase, strip leading "the ", underscores for spaces
            key = raw_name.lower()
            key = re.sub(r"^the\s+", "", key)
            key = key.replace(" ", "_")
            persona_counts[key] += 1
            if key not in persona_system_prompts:
                persona_system_prompts[key] = sys_msg
        else:
            print(f"  ⚠️ Could not extract persona from system prompt: {sys_msg[:80]}...")

dataset = HFDataset.from_list(conversations)

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} multi-turn conversations across {len(persona_counts)} personas")
print(f"Extracted {len(persona_system_prompts)} unique system prompts from JSONL")
print(f"Columns: {dataset.column_names}")
print(f"\nPer-persona breakdown:")
for p, c in sorted(persona_counts.items(), key=lambda x: -x[1]):
    print(f"  {p:20s} {c:>5d} conversations")

# Show a sample prompt to verify extraction
sample_key = next(iter(persona_system_prompts))
print(f"\n--- Sample extracted prompt ({sample_key}, first 200 chars) ---")
print(f"  {persona_system_prompts[sample_key][:200]}...")


## 4. Validate & Summarize Dataset

Datagen data is already clean (no artifacts to strip). Verify data quality and show persona distribution.

In [ ]:

bad_examples = []
empty_responses = []
unique_system_prompts = set()

for i, example in enumerate(dataset):
    convs = example["conversations"]
    # Multi-turn ShareGPT: system, then alternating human/gpt pairs
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count ≥3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    # Validate alternating human/gpt after system
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    # Check last GPT response is not empty
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    unique_system_prompts.add(convs[0]["value"])

# Turn-count distribution
from collections import Counter
turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  Unique system prompts: {len(unique_system_prompts)} (should match extracted count: {len(persona_system_prompts)})")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\n⚠️ Bad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_responses:
    print(f"\n⚠️ Filtering {len(empty_responses)} empty responses...")
    good_indices = [i for i in range(len(dataset)) if i not in set(empty_responses)]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

# Persona distribution
print(f"\nPERSONA DISTRIBUTION:")
max_name_len = max(len(n) for n in persona_counts)
for name, count in sorted(persona_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 50) + "▌" * (1 if count % 50 >= 25 else 0)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(persona_counts.values()):>5}")

# Show voice differentiation — first response from different personas
print(f"\nVOICE SAMPLES (first ~100 chars of response):")
seen_personas = set()
for example in dataset:
    system = example["conversations"][0]["value"]
    # Extract persona name from system prompt "You are X, ..."
    name_part = system.split(",")[0].replace("You are ", "")
    if name_part not in seen_personas and len(seen_personas) < 4:
        response_start = example["conversations"][2]["value"][:100]
        print(f"  {name_part}: \"{response_start}...\"")
        seen_personas.add(name_part)

print(f"\n✓ Dataset validated and ready for training")


## 5. Load Model & Tokenizer (BNB NF4 4-bit)

Load the text-only Qwen3.5 9B BNB NF4 base for QLoRA training.

- **DGX Spark (128GB):** ample headroom for training and experimentation


In [ ]:
import os
# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and broken
# torch.compile path. These are conservative — they prevent the slow eager
# Python fallback that some new architectures hit on sm_120. Must be set
# BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Text-only base should return a tokenizer, but keep this robust for wrappers.
if hasattr(tokenizer, "vocab_size"):
    vocab_size = tokenizer.vocab_size
elif hasattr(tokenizer, "tokenizer") and hasattr(tokenizer.tokenizer, "vocab_size"):
    vocab_size = tokenizer.tokenizer.vocab_size
else:
    vocab_size = "unknown"

print(f"✓ Model loaded: {BASE_LLM}")
print(f"  Precision: pre-quantized BNB NF4 4-bit QLoRA")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {vocab_size}")
print(f"  Attn impl: {getattr(model.config, '_attn_implementation', 'unknown')}")


## 6. Format Dataset for Chat Template

Apply Qwen's tokenizer chat template (ChatML) to each conversation and build the final training dataset.

In [ ]:
# Standardize ShareGPT rows, then format with Qwen tokenizer chat template
from unsloth.chat_templates import standardize_sharegpt

original_dataset_len = len(dataset)

dataset = standardize_sharegpt(dataset)
formatted_texts = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
)

# Build final dataset
import pandas as pd
from datasets import Dataset as HFDataset

dataset = HFDataset.from_pandas(pd.DataFrame({"text": formatted_texts}))

# Filter out empty examples, shuffle deterministically, then optionally cap.
dataset = dataset.filter(lambda x: len(x["text"]) > 0)
filtered_dataset_len = len(dataset)
dataset = dataset.shuffle(seed=42)

if SFT_MAX_EXAMPLES:
    if SFT_MAX_EXAMPLES <= 0:
        raise ValueError(f"SFT_MAX_EXAMPLES must be 0 or positive, got {SFT_MAX_EXAMPLES}")
    if len(dataset) > SFT_MAX_EXAMPLES:
        print(f"Capping SFT dataset from {len(dataset)} to {SFT_MAX_EXAMPLES} examples")
        dataset = dataset.select(range(SFT_MAX_EXAMPLES))
    else:
        print(f"SFT_MAX_EXAMPLES={SFT_MAX_EXAMPLES}, but dataset only has {len(dataset)} examples; using all")

print(f"--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n✓ Dataset formatted: {len(dataset)} examples")
print(f"  Original conversations: {original_dataset_len}")
print(f"  After empty filtering:  {filtered_dataset_len}")
print(f"  SFT cap:                {SFT_MAX_EXAMPLES or 'ALL'}")

## 7. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See Step 1 config for module options.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = trainable_params / total_params * 100

print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"Target modules: {LORA_TARGET_MODULES}")
print(f"Trainable params: {trainable_params:,}")
print(f"Total params:     {total_params:,}")
print(f"Trainable pct:    {trainable_pct:.4f}%")

## 8. Trainer Setup

- 1 epoch, low learning rate (1e-4) — gently teaches persona-switching without degrading base capabilities
- Each example has a persona-specific system prompt so the model learns distinct voices

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=True,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=12,
        report_to="none",
    ),
)

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
print(f"✓ Trainer configured")
print(f"  Effective batch size: {BATCH_SIZE} × {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: enabled")
print(f"  Checkpoints: every 25 steps, keeping last 12")
print(f"  Checkpoint dir: {OUTPUT_DIR_ADAPTERS}")
print(f"  Dataset: {len(dataset)} examples")

## 9. Train

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(OUTPUT_DIR_ADAPTERS)
if last_checkpoint is not None:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting training from scratch; no checkpoint found.")
    trainer.train()

## 10. Save LoRA Adapters

Save the trained LoRA adapters. These can be loaded on compatible Qwen3.5 9B models with PEFT or served via vLLM.

In [ ]:
from pathlib import Path
import json

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Save LoRA adapters + tokenizer
print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

example_persona = next(iter(persona_system_prompts), "marcus_aurelius")

print(f"\n✓ LoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(persona_system_prompts)} personas)")
print(f"\n  At inference, load prompts with:")
print(f'    with open("{prompts_path}") as f:')
print(f'        prompts = json.load(f)')
print(f'    system_msg = prompts["{example_persona}"]  # or any persona key')

## 11. Test Inference

Quick smoke test with a few personas using their extracted system prompts. Each persona should respond in its distinctive voice.


In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Pick up to 4 personas to test
test_personas = list(persona_system_prompts.keys())[:4]

print(f"INFERENCE TEST — {len(test_personas)} PERSONAS\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text=text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  PERSONA: {persona_key.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    # Qwen3.5 recommended sampling (non-thinking text mode):
    # temperature=1.0, top_p=1.0, top_k=20 (presence_penalty is serving-side only)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=1.0,
        top_p=1.0,
        top_k=20,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

## 12. Verify Adapter (Reload from Disk)

Final validation: load the adapter cold from disk to confirm it's self-contained and portable.


In [ ]:
# Clean up training model
import gc, torch
del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("✓ Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

# Reload from disk
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

# Reload saved system prompts
import json
with open(f"{LORA_OUTPUT_DIR}/persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

# Test with first persona
test_key = list(reloaded_prompts.keys())[0]
test_prompt_text = reloaded_prompts[test_key]

messages = [
    {"role": "system", "content": test_prompt_text},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer2(text=text, return_tensors="pt").to(model2.device)

# Qwen3.5 recommended sampling (non-thinking text mode):
# temperature=1.0, top_p=1.0, top_k=20 (presence_penalty is serving-side only)
outputs = model2.generate(
    **inputs,
    max_new_tokens=256,
    temperature=1.0,
    top_p=1.0,
    top_k=20,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (persona: {test_key}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\n✓ Adapter loads cleanly from disk. Ready for deployment via vLLM.")

# List adapter files
print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

## 13. Export Merged Model to GGUF (Mobile / Cross-Platform)

Merge the LoRA adapter into the base model and export to GGUF so it can
run anywhere llama.cpp / Ollama / LM Studio / iOS GGUF runners are supported
(e.g. iPhone via apps like LLMFarm, PocketPal, Private LLM).

- `q4_k_m` is the recommended quant for phones (best size/quality tradeoff).
- `q5_k_m` and `q8_0` are also produced for higher-quality desktop use.
- Requires Unsloth's bundled `llama.cpp` build (downloaded automatically on
  first call to `save_pretrained_gguf`).

**Note:** Qwen3.5 GGUF export requires a recent llama.cpp version that
supports the Qwen3.5 architecture (Gated DeltaNet + Gated Attention hybrid).
If conversion fails, update llama.cpp or wait for upstream support to land.

In [ ]:
# Compatibility shim for PEFT 0.19 + GPTQModel 7.0.0 during GGUF export.
# PEFT imports GPTQModel's older AWQ class name even when this adapter is not AWQ.
try:
    import gptqmodel.nn_modules.qlinear.gemm_awq as _gemm_awq
    if not hasattr(_gemm_awq, "AwqGEMMQuantLinear") and hasattr(_gemm_awq, "AwqGEMMLinear"):
        _gemm_awq.AwqGEMMQuantLinear = _gemm_awq.AwqGEMMLinear
        print("Patched GPTQModel AWQ class alias for PEFT compatibility.")
except Exception as exc:
    print(f"GPTQModel AWQ compatibility shim skipped: {exc}")

In [ ]:
# Reload adapter fresh so we export from a clean state
import gc, torch
from pathlib import Path
from unsloth import FastLanguageModel

try:
    del model2, tokenizer2
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

GGUF_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/gguf"
Path(GGUF_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Load base + adapter; Unsloth's GGUF exporter will merge before conversion
model_gguf, tokenizer_gguf = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # full-precision merge for accurate GGUF quantization
)

# Force the chat-template's end-of-turn token as the GGUF EOS.
#
# Qwen ChatML uses `<|im_end|>` as the closer of `<|im_start|>role\n...content...<|im_end|>\n`.
# Without forcing this, llama.cpp's GGUF converter may inherit a generic `<eos>`
# from `text_config.eos_token_id` on Conditional-Generation models,
# so the model never stops on `<|im_end|>` and emits the next turn header.
# iOS GGUF runners then strip the leading `<|im_start|>` special token and
# render the bare role text ("user\n...") plus a hallucinated user question.
_render = tokenizer_gguf.apply_chat_template(
    [{"role": "user", "content": "x"}, {"role": "assistant", "content": "y"}],
    tokenize=False,
)
_eot_candidates = ["<|im_end|>", "<turn|>", "<end_of_turn>", "<|eot_id|>"]
eot_token = next((c for c in _eot_candidates if c in _render), None)
if eot_token is None:
    raise RuntimeError(f"Could not detect end-of-turn marker in chat template. Rendered: {_render!r}")
# Some loaders wrap the tokenizer; unwrap so we can call tokenizer methods.
_tok = getattr(tokenizer_gguf, "tokenizer", tokenizer_gguf)
eot_id = _tok.convert_tokens_to_ids(eot_token)
if eot_id is None or eot_id == _tok.unk_token_id:
    raise RuntimeError(f"{eot_token!r} not in tokenizer vocab (got id={eot_id})")

_tok.eos_token = eot_token
model_gguf.config.eos_token_id = eot_id
# Qwen3.5 9B is a Conditional-Generation model with a nested text_config; the
# llama.cpp converter reads from there, so we MUST set it here too.
if hasattr(model_gguf.config, "text_config") and model_gguf.config.text_config is not None:
    model_gguf.config.text_config.eos_token_id = eot_id
# generation_config.eos_token_id may be a list; collapse to the single
# chat-template EOS so runners that only honor a scalar pick the right one.
if getattr(model_gguf, "generation_config", None) is not None:
    model_gguf.generation_config.eos_token_id = eot_id

print(f"\u2713 Forced GGUF EOS to {eot_token!r} (id={eot_id})")

# Pass ALL quant methods in one call so the LoRA→FP16 merge happens ONCE
# and llama.cpp quantizes from that single merged file. Otherwise the loop
# re-merges and re-writes the FP16 GGUF for every quant level.
QUANT_METHODS = ["q4_k_m", "q5_k_m", "q8_0"]

print(f"Exporting GGUF (single merge → {len(QUANT_METHODS)} quants)...")
model_gguf.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer_gguf,
    quantization_method=QUANT_METHODS,
)

print(f"\n\u2713 GGUF export complete: {GGUF_OUTPUT_DIR}")
for f in sorted(Path(GGUF_OUTPUT_DIR).glob("*.gguf")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:60s} {size_mb:>8.1f} MB")

print("\nMobile usage (iPhone):")
print("  1. Transfer the q4_k_m .gguf file to your phone (AirDrop / Files app).")
print("  2. Open in an iOS GGUF runner (LLMFarm, PocketPal, Private LLM, etc.).")
print("  3. Use Qwen ChatML template; set context length <= MAX_SEQ_LENGTH.")

del model_gguf, tokenizer_gguf
gc.collect()
torch.cuda.empty_cache()